In [7]:
import pandas as pd
import numbers as np


In [ ]:
#Importation du fichier "Children's record"
df = pd.read_stata("MLKR83FL.DTA")
print("Shape :", df.shape)
print(df.head())

Shape : (9464, 631)
            caseid  bidx v000  v001  v002  v003  v004     v005  v006  v007  \
0         1   1  3     1  ML7     1     1     3     1  1224388    10  2021   
1         1   3  2     1  ML7     1     3     2     1  1224388    10  2021   
2         1   3  2     2  ML7     1     3     2     1  1224388    10  2021   
3         1   3  4     1  ML7     1     3     4     1  1224388    10  2021   
4         1   3  4     2  ML7     1     3     4     1  1224388    10  2021   

   ...  s415da  s415db  s415dc  s415f  s415h  s415i s415j s415m s415n  s415o  
0  ...     NaN     NaN     NaN    NaN    NaN    NaN   NaN   NaN   NaN    NaN  
1  ...     NaN     NaN     NaN    NaN    NaN    NaN   NaN   NaN   NaN    NaN  
2  ...     NaN     NaN     NaN    NaN    NaN    NaN   NaN   NaN   NaN    NaN  
3  ...     NaN     NaN     NaN    NaN    NaN    NaN   NaN   NaN   NaN    NaN  
4  ...     NaN     NaN     NaN    NaN    NaN    NaN   NaN   NaN   NaN    NaN  

[5 rows x 631 columns]


In [5]:
#Extration des noms des colonnes des métadonnées
chemin = r"C:\Users\bobma\OneDrive\Dokumente\Projet_soutenance\MLKR83FL.DTA"

# Charger avec les métadonnées
df = pd.read_stata(chemin, convert_categoricals=False)
meta = pd.read_stata(chemin, iterator=True)

# Extraire le dictionnaire variable → label
variable_labels = meta.variable_labels()

# Afficher toutes les variables avec leur description
dictionnaire = pd.DataFrame(
    list(variable_labels.items()),
    columns=['code_variable', 'description']
)

print(dictionnaire.to_string())

# Exportation en CSV 
dictionnaire.to_csv('dictionnaire_KR_DHS_Mali.csv', index=False, encoding='utf-8-sig')
print("\nFichier dictionnaire_KR_DHS_Mali.csv exporté !")

    code_variable                                                                       description
0          caseid                                                               case identification
1            bidx                                                               birth column number
2            v000                                                            country code and phase
3            v001                                                                    cluster number
4            v002                                                                  household number
5            v003                                                          respondent's line number
6            v004                                                                ultimate area unit
7            v005                                     women's individual sample weight (6 decimals)
8            v006                                                                month of interview


In [13]:
#Importation du fichier "Household member record"
df = pd.read_stata("MLPR83FL.DTA")
print("Shape :", df.shape)
print(df.head())

Shape : (54419, 311)
           hhid  hvidx hv000  hv001  hv002 hv003  hv004    hv005  hv006  \
0         1   1      1   ML7      1      1     1      1  1211097     10   
1         1   1      2   ML7      1      1     1      1  1211097     10   
2         1   1      3   ML7      1      1     1      1  1211097     10   
3         1   1      4   ML7      1      1     1      1  1211097     10   
4         1   1      5   ML7      1      1     1      1  1211097     10   

   hv007  ...  hml32b  hml32c  hml32d  hml32e  hml32f  hml32g  hml33  hml34  \
0   2021  ...     NaN     NaN     NaN     NaN     NaN     NaN    NaN          
1   2021  ...     NaN     NaN     NaN     NaN     NaN     NaN    NaN          
2   2021  ...     NaN     NaN     NaN     NaN     NaN     NaN    NaN          
3   2021  ...     NaN     NaN     NaN     NaN     NaN     NaN    NaN          
4   2021  ...     NaN     NaN     NaN     NaN     NaN     NaN    NaN          

  hml35  hml36  
0   NaN    NaN  
1   NaN    NaN  
2 

In [14]:
#Extration des noms des colonnes des métadonnées
chemin = r"C:\Users\bobma\OneDrive\Dokumente\Projet_soutenance\MLPR83FL.DTA"

# Charger avec les métadonnées
df = pd.read_stata(chemin, convert_categoricals=False)
meta = pd.read_stata(chemin, iterator=True)

# Extraire le dictionnaire variable → label
variable_labels = meta.variable_labels()

# Afficher toutes les variables avec leur description
dictionnaire = pd.DataFrame(
    list(variable_labels.items()),
    columns=['code_variable', 'description']
)

print(dictionnaire.to_string())

# Exportation en CSV 
dictionnaire.to_csv('dictionnaire_PR_DHS_Mali.csv', index=False, encoding='utf-8-sig')
print("\nFichier dictionnaire_PR_DHS_Mali.csv exporté !")

    code_variable                                                                  description
0            hhid                                                          case identification
1           hvidx                                                                  line number
2           hv000                                                       country code and phase
3           hv001                                                               cluster number
4           hv002                                                             household number
5           hv003                 respondent's line number (answering household questionnaire)
6           hv004                                                           ultimate area unit
7           hv005                                         household sample weight (6 decimals)
8           hv006                                                           month of interview
9           hv007                                 

In [16]:

# --- 1. Charger les variables utiles du fichier KR ---
# convert_categoricals=False pour les cles de fusion (sinon erreur de type)
kr_keys = pd.read_stata("MLKR83FL.DTA", columns=['v001','v002','b16'], convert_categoricals=False)
kr_vals = pd.read_stata("MLKR83FL.DTA", columns=['h22','hw1','hw57'], convert_categoricals=True)
kr = pd.concat([kr_keys, kr_vals], axis=1)

# --- 2. Charger les variables utiles du fichier PR ---
pr = pd.read_stata("MLPR83FL.DTA",
                    columns=['hv001','hv002','hvidx','hv024','hv025','hv270','hv104','hml35'],
                    convert_categoricals=True)
pr = pr.rename(columns={'hv024':'region','hv025':'milieu','hv270':'richesse','hv104':'sexe'})

# --- 3. Fusion (inner join) sur les cles communes ---
# v001/v002/b16 (cote KR) correspondent a hv001/hv002/hvidx (cote PR) : meme personne, nomenclature differente
df = kr.merge(pr, left_on=['v001','v002','b16'], right_on=['hv001','hv002','hvidx'], how='inner')
df = df.rename(columns={'h22':'fievre','hw1':'age_mois','hw57':'anemie','hml35':'resultat_test'})

# --- 4. Ne garder que les resultats de test exploitables (positif/negatif) ---
df_final = df[df['resultat_test'].isin(['positive','negative'])].copy()

# --- 5. Selectionner les colonnes finales et retirer les valeurs manquantes ---
cols_finales = ['region','milieu','richesse','sexe','age_mois','fievre','anemie','resultat_test']
df_final = df_final[cols_finales]

print("Shape final:", df_final.shape)
#print(df_final['resultat_test'].value_counts(normalize=True)*100)

df_final.to_csv("dataset_paludisme_mali_final.csv", index=False, encoding='utf-8-sig')

Shape final: (7859, 8)
